# Hydro displacement of natural gas in Tanzania's power sector
**Interrupted time series around the commissioning of the Julius Nyerere Hydropower Project (JNHPP)**

Author: Abbot Josephat Antony

**Question.** Did gas burned for electricity fall when JNHPP came online, by how much, and are the gas and electricity statistics physically consistent with that story?

**Data.** Bank of Tanzania *Consolidated Zonal Economic Reports* (scraped): quarterly gas production, gas consumption by sector, and electricity generation by source. JNHPP turbine timeline compiled from the Daily News corpus.

**Methods.**
1. Edition-aware cleaning of the BoT quarterly tables
2. Interrupted time series (segmented regression, Newey–West SEs)
3. Dose-response on the number of JNHPP turbines operating
4. Physical consistency checks (implied heat rate; gas displaced per GWh of hydro)
5. Unknown-break scan (sup-F) on the long production series

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Paths are relative to the repository root (where this notebook sits)
DATA_DIR    = Path("data/raw")       # bot_gas.csv, bot_electricity.csv (tracked in git)
PRIVATE_DIR = Path("data/private")   # copyrighted news text, NOT tracked in git
OUT         = Path("outputs")        # all tables and figures are saved here
OUT.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", OUT)

NW_LAGS   = 2         # Newey-West lags (quarterly data)
BREAK     = "2024Q2"  # first full quarter after JNHPP turbine 9 (22 Feb 2024)
ALT_BREAK = "2024Q1"  # robustness: quarter of first grid injection

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 2. Load the scraped BoT tables

In [ ]:
gas = pd.read_csv(DATA_DIR / "bot_gas.csv", low_memory=False)
ele = pd.read_csv(DATA_DIR / "bot_electricity.csv", low_memory=False)

print("bot_gas:", gas.shape, "| variables:", gas.variable.unique().tolist())
print("bot_electricity:", ele.shape, "| variables:", ele.variable.unique().tolist())
gas.loc[gas.month.notna(), ["year", "month", "variable", "category", "value_mmscf", "doc_title", "latest_edition"]].head()

## 3. Edition-aware selection of quarterly values

The same quarter appears in several report editions, and editions sometimes disagree. Two problems matter here:

* **Carry-forward errors** — an edition reprints the previous quarter's numbers (e.g. Dec-2022 gas consumption in the Dec-2023 report equals Sep-2022 exactly).
* **Revisions** — later editions change earlier figures, sometimes by a lot.

**Rule:** use the latest edition, *unless* its value equals the previously selected quarter and another edition gives a different value; then use that edition. Every intervention and every >5% disagreement between editions is written to a log.

In [ ]:
issues = []

def pick_quarterly(df, value_col, label):
    df = df[df.month.notna()].copy()
    df["q"] = pd.PeriodIndex.from_fields(year=df.year.astype(int),
                                         quarter=(df.month.astype(int) - 1) // 3 + 1, freq="Q")
    df["pub_date"] = pd.to_datetime(df.pub_date, errors="coerce")
    rows = []
    for cat, g in df.groupby("category"):
        prev = None
        for q, gq in g.sort_values("q").groupby("q"):
            gq = gq.dropna(subset=[value_col]).sort_values(["latest_edition", "pub_date"], ascending=False)
            if gq.empty:
                continue
            chosen = gq.iloc[0]
            if prev is not None and np.isclose(chosen[value_col], prev):
                alt = gq[~np.isclose(gq[value_col], prev)]
                if len(alt):
                    issues.append(dict(series=label, category=cat, quarter=str(q),
                                       issue="latest edition repeats previous quarter; replaced",
                                       value_latest=chosen[value_col], value_used=alt.iloc[0][value_col],
                                       source_used=alt.iloc[0].doc_title))
                    chosen = alt.iloc[0]
                else:
                    issues.append(dict(series=label, category=cat, quarter=str(q),
                                       issue="equals previous quarter in all editions; kept",
                                       value_latest=chosen[value_col], value_used=chosen[value_col],
                                       source_used=chosen.doc_title))
            vals = gq[value_col].unique()
            if len(vals) > 1 and (vals.max() - vals.min()) / vals.max() > 0.05:
                issues.append(dict(series=label, category=cat, quarter=str(q),
                                   issue=f"editions disagree >5% ({vals.min():,.1f} to {vals.max():,.1f})",
                                   value_latest=gq.iloc[0][value_col], value_used=chosen[value_col],
                                   source_used=chosen.doc_title))
            rows.append(dict(q=q, category=cat, value=chosen[value_col]))
            prev = chosen[value_col]
    return pd.DataFrame(rows).pivot(index="q", columns="category", values="value")

cons = pick_quarterly(gas[gas.variable == "gas_consumption"], "value_mmscf", "gas_consumption")
prod = pick_quarterly(gas[(gas.variable == "gas_production") & (gas.category == "TOTAL")], "value_mmscf", "gas_production")
elec = pick_quarterly(ele[(ele.variable == "electricity_generation_by_source") &
                          (ele.x_period_type == "quarter_ending")], "value_mwh", "electricity")
print("Quarters available -> consumption:", len(cons), "| production:", len(prod), "| electricity:", len(elec))

## 4. Build the quarterly panel

In [ ]:
panel = pd.DataFrame(index=pd.period_range("2015Q3", "2026Q1", freq="Q"))
panel.index.name = "period"
panel["gas_production"] = prod["TOTAL"]
panel["gas_power"]      = cons["Power generating plants"]
panel["gas_industry"]   = cons["Industries"]
panel["gas_vehicles"]   = cons["Vehicles"]
panel["gas_total"]      = cons["TOTAL"]

mwh_to_gwh = 1 / 1000
panel["hydro_tanesco_gwh"]   = elec["TANESCO plants: hydro"] * mwh_to_gwh
panel["thermal_tanesco_gwh"] = elec["TANESCO plants: thermal"] * mwh_to_gwh
panel["thermal_private_gwh"] = elec["Private plants: thermal"] * mwh_to_gwh
panel["hydro_private_gwh"]   = elec["Private plants: hydro"] * mwh_to_gwh
panel["imports_gwh"]         = elec["Imported"] * mwh_to_gwh
panel["total_gen_gwh"]       = elec["Total (as printed)"] * mwh_to_gwh
panel["thermal_gwh"]         = panel.thermal_tanesco_gwh + panel.thermal_private_gwh

# 2025Q2 electricity is implausible: total 4.2 TWh (vs ~3.3 either side) and thermal doubles
# while gas-for-power falls. Flag it and exclude from the electricity models.
panel["elec_suspect"] = 0
panel.loc[pd.Period("2025Q2", "Q"), "elec_suspect"] = 1
issues.append(dict(series="electricity", category="all", quarter="2025Q2",
                   issue="implausible spike (total 4.2 TWh, thermal 2.1 TWh); excluded from electricity models",
                   value_latest=np.nan, value_used=np.nan, source_used="Consolidated Zonal Economic Report - Sep-25"))

panel["t"] = np.arange(len(panel)) + 1
panel["quarter"] = panel.index.quarter
panel[["gas_production", "gas_power", "gas_industry", "gas_vehicles", "hydro_tanesco_gwh", "thermal_gwh", "total_gen_gwh"]].tail(10)

## 5. JNHPP turbine timeline (treatment intensity)

Turbines operating at the end of each quarter (235 MW each), compiled from the Daily News corpus:

| Date | Event |
|---|---|
| 22 Feb 2024 | Turbine 9 starts supplying the grid (235 MW) |
| Mar–Apr 2024 | Second unit commissioned |
| 17 Jul 2024 | Third turbine switched on |
| Dec 2024 | Five turbines operational (1,175 MW) |
| Feb 2025 | Eight turbines (1,880 MW) |
| Feb–Mar 2025 | Ninth turbine trials; full 2,115 MW from 2025Q2 |

**Verify against TANESCO / Ministry of Energy statements before publication.** The code below also searches your news corpus so you can check the dates yourself.

In [ ]:
turbines = {"2024Q1": 1, "2024Q2": 2, "2024Q3": 3, "2024Q4": 5, "2025Q1": 8}
panel["jnhpp_turbines"] = 0
for q, n in turbines.items():
    panel.loc[pd.Period(q, "Q"), "jnhpp_turbines"] = n
panel.loc[panel.index >= pd.Period("2025Q2", "Q"), "jnhpp_turbines"] = 9
panel["jnhpp_mw"] = panel.jnhpp_turbines * 235
panel.loc["2023Q4":, ["jnhpp_turbines", "jnhpp_mw"]]

In [ ]:
# Check the timeline against the news corpus
# Optional: runs only if you have the news corpus locally in data/private/
news_path = PRIVATE_DIR / "news_dailynews.csv"
if news_path.exists():
    news = pd.read_csv(news_path)
    hits = news[news.x_text.str.contains("turbine", case=False, na=False)
                & news.x_text.str.contains("Nyerere|JNHPP", case=False, na=False)]
    hits = hits[(hits.pub_date >= "2024-01") & (hits.pub_date <= "2025-06")]
    print(len(hits), "articles mention JNHPP turbines, Jan 2024 - Jun 2025")
    display(hits[["pub_date", "doc_title"]].sort_values("pub_date").head(20))
else:
    print("News corpus not found in data/private/ - skipping the timeline check.")

## 6. Data-issue log

In [ ]:
issues_df = pd.DataFrame(issues)
issues_df.to_csv(OUT / "data_issues_log.csv", index=False)
print(issues_df.issue.str.split(" \\(").str[0].value_counts())
issues_df[issues_df.category.isin(["Power generating plants", "TOTAL", "TANESCO plants: hydro", "TANESCO plants: thermal", "all"])]

## 7. Descriptive look

In [ ]:
xs = lambda idx: idx.to_timestamp(how="end")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

d = panel.dropna(subset=["gas_total"])
ax[0].plot(xs(d.index), d.gas_power, "o-", label="Power")
ax[0].plot(xs(d.index), d.gas_industry, "s-", label="Industry")
ax[0].plot(xs(d.index), d.gas_vehicles * 10, "^-", label="Vehicles (x10)")
ax[0].axvline(pd.Timestamp("2024-02-22"), color="green", ls=":", label="JNHPP first turbine")
ax[0].set_title("Gas consumption by sector")
ax[0].set_xlabel("Time"); ax[0].set_ylabel("Gas consumption (MMscf per quarter)")
ax[0].legend(fontsize=8)

e = panel[panel.elec_suspect == 0].dropna(subset=["total_gen_gwh"])
ax[1].stackplot(xs(e.index), e.hydro_tanesco_gwh, e.hydro_private_gwh.fillna(0), e.thermal_tanesco_gwh,
                e.thermal_private_gwh, e.imports_gwh.fillna(0),
                labels=["TANESCO hydro", "Private hydro", "TANESCO thermal", "Private thermal", "Imports"],
                colors=["#2e75b6", "#9dc3e6", "#c55a11", "#f4b183", "#a5a5a5"])
ax[1].set_title("Electricity generation by source")
ax[1].set_xlabel("Time"); ax[1].set_ylabel("Electricity generation (GWh per quarter)")
ax[1].legend(fontsize=8, loc="upper left")
plt.tight_layout(); fig.savefig(OUT / "descriptive_gas_and_electricity.png", dpi=200); plt.show()

## 8. Interrupted time series

$$y_t = \beta_0 + \beta_1 t + \beta_2\,\text{post}_t + \beta_3\,(t - T_0 + 1)\,\text{post}_t + \varepsilon_t$$

* $\beta_2$ = immediate level shift at the break; $\beta_3$ = change in quarterly trend after it.
* Counterfactual = the fitted model with $\text{post}=0$ (pre-break trend extended).
* Newey–West SEs with 2 lags. Variants: logs, quarter fixed effects (seasonality from the rains), and an earlier break (2024Q1).
* Two outcomes: **gas for power** (direct, 18 quarters) and **total gas production** (proxy, 42 quarters).

In [ ]:
def its(df, y, brk, log=False, seasonal=False):
    d = df.dropna(subset=[y]).copy()
    T0 = df.loc[pd.Period(brk, "Q"), "t"]
    d["post"] = (d.t >= T0).astype(int)
    d["t_post"] = np.where(d.post == 1, d.t - T0 + 1, 0)
    d["dep"] = np.log(d[y]) if log else d[y]
    f = "dep ~ t + post + t_post" + (" + C(quarter)" if seasonal else "")
    m = smf.ols(f, d).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
    d["fitted"] = m.predict(d)
    d["counterfactual"] = m.predict(d.assign(post=0, t_post=0))
    if log:
        d["fitted"], d["counterfactual"] = np.exp(d.fitted), np.exp(d.counterfactual)
    p = d[d.post == 1]
    gap = (p.counterfactual - p[y]).sum()
    return m, d, gap, gap / p.counterfactual.sum()

m_main, d_main, gap, gap_pct = its(panel, "gas_power", BREAK)
print(m_main.summary())
(OUT / "its_main_model_summary.txt").write_text(m_main.summary().as_text())
print(f"\nGas for power displaced since {BREAK}: {gap:,.0f} MMscf ({gap_pct:.1%} below counterfactual)")

### 8.1 Robustness across specifications

In [ ]:
specs = [(y, b, lg, ss) for y in ["gas_power", "gas_production"]
         for (b, lg, ss) in [(BREAK, False, False), (BREAK, True, False), (BREAK, False, True), (ALT_BREAK, False, False)]]
rows, fits = [], {}
for y, b, lg, ss in specs:
    m, d, g, gp = its(panel, y, b, lg, ss)
    fits[(y, b, lg, ss)] = d
    rows.append(dict(outcome=y, brk=b, spec=("log" if lg else "level") + (" + quarter FE" if ss else ""), n=int(m.nobs),
                     level_shift=m.params["post"], p_level=m.pvalues["post"],
                     slope_change=m.params["t_post"], p_slope=m.pvalues["t_post"],
                     displaced_mmscf=g, displaced_pct=gp))
its_table = pd.DataFrame(rows)
its_table.to_csv(OUT / "its_results.csv", index=False)
its_table.style.format({"level_shift": "{:,.3f}", "slope_change": "{:,.3f}", "p_level": "{:.3f}",
                        "p_slope": "{:.3f}", "displaced_mmscf": "{:,.0f}", "displaced_pct": "{:.1%}"})

### 8.2 Actual vs counterfactual

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for a, y, title in [(ax[0], "gas_power", "Gas consumed by power plants"),
                    (ax[1], "gas_production", "Total gas production (long series)")]:
    d = fits[(y, BREAK, False, False)]
    p = d[d.post == 1]
    a.plot(xs(d.index), d[y], "o-", ms=4, color="#1f4e79", label="Actual")
    a.plot(xs(d.index), d.fitted, color="#1f4e79", alpha=.35, label="ITS fit")
    a.plot(xs(p.index), p.counterfactual, "--", color="#c0504d", label="Counterfactual")
    a.fill_between(xs(p.index), p[y], p.counterfactual, color="#c0504d", alpha=.12)
    a.axvline(pd.Timestamp("2024-02-22"), color="green", ls=":", label="JNHPP first turbine")
    a.set_title(title); a.set_xlabel("Time"); a.set_ylabel("Gas volume (MMscf per quarter)"); a.legend(fontsize=8)
ax[0].set_xlim(pd.Timestamp("2020-07-01"), pd.Timestamp("2026-06-30"))
plt.tight_layout(); fig.savefig(OUT / "its_counterfactual.png", dpi=200); plt.show()

## 9. Dose-response: gas use and the number of JNHPP turbines

Instead of a single on/off break, use the number of turbines operating as a treatment-intensity variable, controlling for the underlying trend.

In [ ]:
rows = []
for y in ["gas_power", "gas_production"]:
    for ss in [False, True]:
        d = panel.dropna(subset=[y])
        m = smf.ols(f"{y} ~ jnhpp_turbines + t" + (" + C(quarter)" if ss else ""), d).fit(
            cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
        rows.append(dict(outcome=y, quarter_FE=ss, n=int(m.nobs), per_turbine_mmscf=m.params["jnhpp_turbines"],
                         se=m.bse["jnhpp_turbines"], p=m.pvalues["jnhpp_turbines"]))
dose = pd.DataFrame(rows)
dose.to_csv(OUT / "dose_response_results.csv", index=False)
dose

In [ ]:
# Benchmark: what would one fully loaded turbine displace?
heat_rate_guess = 9.5                       # MMscf per GWh (estimated in section 10)
for cf in [0.3, 0.5, 0.7]:
    gwh = 235 * cf * 2190 / 1000            # 2,190 hours per quarter
    print(f"capacity factor {cf:.0%}: {gwh:,.0f} GWh/quarter -> {gwh*heat_rate_guess:,.0f} MMscf/quarter")

## 10. Physical consistency checks

If the gas and electricity tables are right, (a) gas for power should track thermal generation at a realistic **heat rate**, and (b) each GWh of hydro (holding total generation fixed) should displace roughly that much gas.

Benchmark: modern combined-cycle plants need about 6.5–7 MMscf per GWh; open-cycle and older plants roughly 9–11 MMscf per GWh (at ~1,037 Btu/scf).

In [ ]:
d = panel[panel.elec_suspect == 0].dropna(subset=["gas_power", "thermal_gwh", "hydro_tanesco_gwh"])
m_hr  = smf.ols("gas_power ~ thermal_gwh", d).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
m_own = smf.ols("gas_power ~ thermal_tanesco_gwh + thermal_private_gwh", d).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
m_ds  = smf.ols("gas_power ~ hydro_tanesco_gwh + total_gen_gwh", d).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})

print(f"n = {int(m_hr.nobs)} quarters")
print(f"Implied heat rate:            {m_hr.params['thermal_gwh']:.2f} MMscf/GWh  (p = {m_hr.pvalues['thermal_gwh']:.4f}, R2 = {m_hr.rsquared:.2f})")
print(f"Gas displaced per GWh hydro:  {m_ds.params['hydro_tanesco_gwh']:.2f} MMscf/GWh  (p = {m_ds.pvalues['hydro_tanesco_gwh']:.4f})")
print("\nBy plant owner:"); print(m_own.summary().tables[1])

(OUT / "physical_consistency_models.txt").write_text(
    "HEAT RATE\n" + m_hr.summary().as_text() + "\n\nBY OWNER\n" + m_own.summary().as_text()
    + "\n\nDISPLACEMENT\n" + m_ds.summary().as_text())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(d.thermal_gwh, d.gas_power, c=(d.index >= pd.Period(BREAK, "Q")), cmap="coolwarm")
xx = np.linspace(d.thermal_gwh.min(), d.thermal_gwh.max(), 50)
ax.plot(xx, m_hr.params["Intercept"] + m_hr.params["thermal_gwh"] * xx, color="grey")
for q, r in d.iterrows():
    ax.annotate(str(q), (r.thermal_gwh, r.gas_power), fontsize=7, alpha=.6)
ax.set_xlabel("Thermal generation (GWh per quarter)"); ax.set_ylabel("Gas for power (MMscf per quarter)")
ax.set_title("Gas use vs thermal generation (red = after break)"); fig.savefig(OUT / "heat_rate_scatter.png", dpi=200, bbox_inches="tight"); plt.show()

## 11. Where does the data put the break? (sup-F scan)

Fit the segmented model at every candidate break date (15% trimming) on the 42-quarter production series and compute the Chow F-statistic against a no-break trend. The maximum (sup-F) identifies the most likely break. Compare with Andrews' (1993) critical values for two restrictions at 15% trimming (≈11.7 at 5%).

In [ ]:
d = panel.dropna(subset=["gas_production"]).copy()
n = len(d)
base = smf.ols("gas_production ~ t", d).fit()
scan = []
for i in range(int(0.15 * n), int(0.85 * n)):
    T0 = d.t.iloc[i]
    dd = d.assign(post=(d.t >= T0).astype(int))
    dd["t_post"] = np.where(dd.post == 1, dd.t - T0 + 1, 0)
    m = smf.ols("gas_production ~ t + post + t_post", dd).fit()
    scan.append(dict(break_quarter=d.index[i], F=((base.ssr - m.ssr) / 2) / (m.ssr / m.df_resid)))
scan = pd.DataFrame(scan)
best = scan.loc[scan.F.idxmax()]
print(f"sup-F = {best.F:.1f} at {best.break_quarter}")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(xs(pd.PeriodIndex(scan.break_quarter)), scan.F, "o-", ms=3)
ax.axhline(11.7, color="red", ls="--", label="approx. 5% critical value")
ax.set_xlabel("Time"); ax.set_ylabel("Chow F-statistic"); ax.set_title("Break-date scan, total gas production"); ax.legend()
fig.savefig(OUT / "break_scan_plot.png", dpi=200, bbox_inches="tight"); plt.show()
scan.to_csv(OUT / "break_scan_gas_production.csv", index=False)

## 12. Save the panel

In [ ]:
panel.to_csv(OUT / "tz_quarterly_gas_power_panel.csv")
print("All outputs saved to", OUT)
for f in sorted(OUT.iterdir()):
    print("  ", f.name)

## 13. Interpretation notes and limitations

* **Short direct series.** Gas-for-power has 18 quarters (10 before, 8 after the break). Treat effect sizes as indicative; the 42-quarter production series is the robustness anchor.
* **Counterfactual is an upper bound.** It extends the 2020–2024 growth trend; pipeline and processing capacity would eventually have capped growth.
* **Partial dose.** Per-turbine displacement is well below a fully loaded unit, consistent with reports that some turbines were held back because demand could not absorb supply.
* **Classification changes.** Private thermal generation collapses after 2024 and early editions put almost all TANESCO output under hydro for Dec-2024 and Mar-2025. Check whether these are real shifts or reclassification before interpreting them.
* **Turbine dates** come from news reports and must be confirmed with official sources.

**Next steps:** value the displaced gas (lost gas-sector revenue, idle pipeline capacity), and add the industrial/CNG components of the diversification paper.